# Baseline Comparison: RF & XGBoost vs ChemBERTa

Traditional ML on molecular fingerprints is a critical sanity check.
On small datasets (< 10k compounds), fingerprint-based models frequently match or beat deep learning.

**Approach:**
- Convert SMILES to **Morgan fingerprints** (ECFP4: radius=2, 2048 bits) using RDKit
- Train **Random Forest** and **XGBoost** per target, masking untested compounds
- Use the **identical splits** as Milestone 2 (same seed, same data)
- Compare ROC-AUC per target against ChemBERTa

**Why Morgan fingerprints?**
ECFP4 encodes circular substructures around each atom up to radius 2, capturing functional
groups and local chemistry. The industry standard for molecular ML baselines.

## Section 1: Setup

In [ ]:
!pip install -q rdkit xgboost scikit-learn datasets numpy pandas

In [ ]:
import numpy as np
import pandas as pd
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from datasets import load_dataset
from rdkit import Chem
from rdkit.Chem import AllChem

SEED = 42

TASK_COLUMNS = [
    "NR-AR", "NR-AR-LBD", "NR-AhR", "NR-Aromatase",
    "NR-ER", "NR-ER-LBD", "NR-PPAR-gamma",
    "SR-ARE", "SR-ATAD5", "SR-HSE", "SR-MMP", "SR-p53",
]

# ChemBERTa Milestone 2 results — used in the final comparison
CHEMBERTA_AUCS = {
    "NR-AR": 0.8319,
    "NR-AR-LBD": 0.8316,
    "NR-AhR": 0.8624,
    "NR-Aromatase": 0.8764,
    "NR-ER": 0.7135,
    "NR-ER-LBD": 0.7315,
    "NR-PPAR-gamma": 0.7594,
    "SR-ARE": 0.7884,
    "SR-ATAD5": 0.8393,
    "SR-HSE": 0.7831,
    "SR-MMP": 0.8784,
    "SR-p53": 0.8506
}

print(f"Libraries loaded. Targets: {len(TASK_COLUMNS)}")

## Section 2: Fingerprints

**Morgan fingerprints (ECFP4):** Each bit represents the presence of a circular substructure
centred on an atom, up to radius 2. This captures functional groups and ring environments.

```
CC(=O)Oc1ccccc1C(=O)O  (Aspirin)
         ↓ RDKit
[0, 1, 0, 0, 1, 1, 0, ..., 1]   # 2048-bit vector
```

In [ ]:
FP_RADIUS = 2
FP_NBITS  = 2048

def smiles_to_fp(smiles):
    """Convert a SMILES string to a Morgan fingerprint bit vector."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, FP_RADIUS, nBits=FP_NBITS)
    return np.array(fp, dtype=np.float32)

demo = {
    "Aspirin":  "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine": "Cn1c(=O)c2c(ncn2C)n(c1=O)C",
    "Ethanol":  "CCO",
}
for name, smi in demo.items():
    fp = smiles_to_fp(smi)
    print(f"{name:<12} {FP_NBITS} bits, {int(fp.sum())} set ({100*fp.sum()/FP_NBITS:.1f}%)")

## Section 3: Data — Same Splits as Milestone 2

In [ ]:
print("Loading Tox21...")
raw_dataset = load_dataset("scikit-fingerprints/MoleculeNet_Tox21")
df_raw = raw_dataset['train'].to_pandas()
SMILES_COL = 'smiles' if 'smiles' in df_raw.columns else 'SMILES'

df = df_raw[df_raw[TASK_COLUMNS].notna().any(axis=1)].copy()
label_matrix = df[TASK_COLUMNS].fillna(-1).values.astype(np.float32)

processed_df = pd.DataFrame({
    'smiles': df[SMILES_COL].values,
    'labels': [row.tolist() for row in label_matrix],
})

stratify_col = (label_matrix.max(axis=1) > 0).astype(int)
train_df, temp_df, _, strat_temp = train_test_split(
    processed_df, stratify_col,
    test_size=0.30, random_state=SEED, stratify=stratify_col,
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=strat_temp,
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

In [ ]:
def compute_fingerprints(df):
    fps, valid = [], []
    for smi in df['smiles']:
        fp = smiles_to_fp(smi)
        if fp is not None:
            fps.append(fp); valid.append(True)
        else:
            fps.append(np.zeros(FP_NBITS, dtype=np.float32)); valid.append(False)
    return np.array(fps), np.array(valid)

X_train, valid_train = compute_fingerprints(train_df)
X_val,   valid_val   = compute_fingerprints(val_df)
X_test,  valid_test  = compute_fingerprints(test_df)

labels_train = np.array(train_df['labels'].tolist())
labels_val   = np.array(val_df['labels'].tolist())
labels_test  = np.array(test_df['labels'].tolist())

print(f"Train fingerprint matrix: {X_train.shape}")
print(f"Invalid SMILES: {(~valid_train).sum() + (~valid_val).sum() + (~valid_test).sum()}")

## Section 4: Random Forest

One classifier per target, trained only on compounds tested for that target.
`class_weight='balanced'` automatically upweights the minority toxic class.

In [ ]:
RF_PARAMS = dict(
    n_estimators=500,
    class_weight='balanced',
    max_features='sqrt',
    min_samples_leaf=2,
    random_state=SEED,
    n_jobs=-1,
)

rf_models     = {}
rf_test_probs = np.zeros((len(test_df), len(TASK_COLUMNS)))

print("Training Random Forest...")
for i, task in enumerate(TASK_COLUMNS):
    mask = (labels_train[:, i] != -1) & valid_train
    X_tr = X_train[mask]
    y_tr = labels_train[mask, i]
    rf = RandomForestClassifier(**RF_PARAMS)
    rf.fit(X_tr, y_tr)
    rf_models[task] = rf
    rf_test_probs[:, i] = rf.predict_proba(X_test)[:, 1]
    print(f"  {task:<20} n={len(X_tr):>4}  pos={int(y_tr.sum())}")
print("Done.")

In [ ]:
rf_aucs = {}
bert_mean = np.mean(list(CHEMBERTA_AUCS.values()))

print(f"{"Target":<20} {"RF AUC":<12} {"ChemBERTa":<12} {"Delta"}")
print("-" * 52)
for i, task in enumerate(TASK_COLUMNS):
    mask = (labels_test[:, i] != -1) & valid_test
    auc  = roc_auc_score(labels_test[mask, i], rf_test_probs[mask, i])
    rf_aucs[task] = auc
    delta  = auc - CHEMBERTA_AUCS[task]
    sign   = '+' if delta >= 0 else ''
    winner = " <- RF wins" if delta > 0.01 else (" <- ChemBERTa" if delta < -0.01 else "  ~ tie")
    print(f"{task:<20} {auc:<12.4f} {CHEMBERTA_AUCS[task]:<12.4f} {sign}{delta:.4f}{winner}")

rf_mean = np.mean(list(rf_aucs.values()))
print(f"\n{"Mean":<20} {rf_mean:<12.4f} {bert_mean:<12.4f} {rf_mean-bert_mean:+.4f}")

## Section 5: XGBoost

Gradient boosted trees built sequentially, each correcting previous errors.
`scale_pos_weight = n_neg / n_pos` per target mirrors the class weighting approach.

In [ ]:
XGB_BASE = dict(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=SEED,
    n_jobs=-1,
    verbosity=0,
)

xgb_models     = {}
xgb_test_probs = np.zeros((len(test_df), len(TASK_COLUMNS)))

print("Training XGBoost...")
for i, task in enumerate(TASK_COLUMNS):
    mask = (labels_train[:, i] != -1) & valid_train
    X_tr = X_train[mask]
    y_tr = labels_train[mask, i]
    n_pos = int(y_tr.sum()); n_neg = int((y_tr == 0).sum())
    spw = n_neg / n_pos if n_pos > 0 else 1.0
    xgb = XGBClassifier(**XGB_BASE, scale_pos_weight=spw)
    xgb.fit(X_tr, y_tr)
    xgb_models[task] = xgb
    xgb_test_probs[:, i] = xgb.predict_proba(X_test)[:, 1]
    print(f"  {task:<20} n={len(X_tr):>4}  pos={n_pos}  spw={spw:.1f}x")
print("Done.")

In [ ]:
xgb_aucs = {}
print(f"{"Target":<20} {"XGB AUC":<12} {"ChemBERTa":<12} {"Delta"}")
print("-" * 52)
for i, task in enumerate(TASK_COLUMNS):
    mask = (labels_test[:, i] != -1) & valid_test
    auc  = roc_auc_score(labels_test[mask, i], xgb_test_probs[mask, i])
    xgb_aucs[task] = auc
    delta  = auc - CHEMBERTA_AUCS[task]
    sign   = '+' if delta >= 0 else ''
    winner = " <- XGB wins" if delta > 0.01 else (" <- ChemBERTa" if delta < -0.01 else "  ~ tie")
    print(f"{task:<20} {auc:<12.4f} {CHEMBERTA_AUCS[task]:<12.4f} {sign}{delta:.4f}{winner}")

xgb_mean = np.mean(list(xgb_aucs.values()))
print(f"\n{"Mean":<20} {xgb_mean:<12.4f} {bert_mean:<12.4f} {xgb_mean-bert_mean:+.4f}")

## Section 6: Full Three-Way Comparison

In [ ]:
print(f"{"Target":<20} {"RF":<10} {"XGBoost":<10} {"ChemBERTa":<12} {"Winner"}")
print("=" * 60)
for task in TASK_COLUMNS:
    rf_a   = rf_aucs.get(task, 0)
    xgb_a  = xgb_aucs.get(task, 0)
    bert_a = CHEMBERTA_AUCS[task]
    best   = max(rf_a, xgb_a, bert_a)
    winner = "RF" if best == rf_a else ("XGB" if best == xgb_a else "ChemBERTa")
    def b(v): return f'[{v:.4f}]' if abs(v - best) < 1e-9 else f' {v:.4f} '
    print(f"{task:<20} {b(rf_a):<10} {b(xgb_a):<10} {b(bert_a):<12} {winner}")

rf_mean  = np.mean(list(rf_aucs.values()))
xgb_mean = np.mean(list(xgb_aucs.values()))
print("=" * 60)
def b(v): return f'[{v:.4f}]' if v == max(rf_mean, xgb_mean, bert_mean) else f' {v:.4f} '
print(f"{"Mean":<20} {b(rf_mean):<10} {b(xgb_mean):<10} {b(bert_mean):<12}")

In [ ]:
rf_wins, xgb_wins, bert_wins = 0, 0, 0
for task in TASK_COLUMNS:
    best = max(rf_aucs[task], xgb_aucs[task], CHEMBERTA_AUCS[task])
    if best == CHEMBERTA_AUCS[task]: bert_wins += 1
    elif best == xgb_aucs[task]:     xgb_wins  += 1
    else:                            rf_wins   += 1

print(f"Targets won (best ROC-AUC per target):")
print(f"  ChemBERTa:     {bert_wins}/12  mean={bert_mean:.4f}")
print(f"  XGBoost:       {xgb_wins}/12   mean={xgb_mean:.4f}")
print(f"  Random Forest: {rf_wins}/12   mean={rf_mean:.4f}")

## Section 7: Feature Importance

One advantage of tree-based models: interpretable feature importance.
The most important fingerprint bits indicate which substructures drive toxicity predictions.

In [ ]:
TOP_N = 5
print(f"Top {TOP_N} fingerprint bits per target (Random Forest):\n")
for task in TASK_COLUMNS:
    imp     = rf_models[task].feature_importances_
    top_idx = np.argsort(imp)[::-1][:TOP_N]
    top_imp = imp[top_idx]
    print(f"{task}: bits {list(top_idx)}")
    print(f"  importance: {[round(x,4) for x in top_imp]}")

## Key Takeaways

- **If ChemBERTa wins:** Pre-training on 77M molecules gives structural knowledge fingerprints can't derive from Tox21 alone.
- **If RF/XGB win:** That target has clear structural rules a 2048-bit fingerprint captures directly — a larger transformer or more data would be needed to close the gap.
- **Ensemble opportunity:** Averaging ChemBERTa + XGBoost probabilities often outperforms either alone — a common production pattern in computational drug discovery.